In [2]:
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers
!pip install torch

In [3]:
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from transformers import pipeline

In [4]:
pdf_path = "data/The-Art-of-War-Sun-Tzu.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    
    page_text = page.extract_text()
    
    if page_text:
        text += page_text

print(text[:1000])

 
 
 
Contents
The Art of War
I. Laying Plans
II. Waging War
III. Attack by Stratagem
IV. Tactical
Dispositions
V. Energy
VI. Weak Points and Strong
VII. Manoeuvring
VIII. Variation of Tactics
IX. The Army on the
March
X. Terrain
XI. The Nine Situations
XII. The Attack by Fire
XIII. The Use of SpiesThe Art of War 
32I. Laying Plans167F
168 
Sun Tzǔ said: The art of war is of vital importance to the State. 
It is a matter of life and death, a road either to safety or to ruin. Hence it is a subject of
inquiry which can on no account be neglected. 
The art of war, then, is governed by five constant factors, to be taken into account in one’s
deliberations, when seeking to determine the conditions obtaining in the field.168F
169  
These are: (1) The Moral Law; (2) Heaven; (3) Earth; (4) The Commander; (5) Method and
discipline.169F
170  
The Moral Law causes the people to be in complete accord with their ruler, so that they will
follow him regardless of their lives, undismayed by any danger

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))

print(chunks[0])

Number of chunks: 279
Contents
The Art of War
I. Laying Plans
II. Waging War
III. Attack by Stratagem
IV. Tactical
Dispositions
V. Energy
VI. Weak Points and Strong
VII. Manoeuvring
VIII. Variation of Tactics
IX. The Army on the
March
X. Terrain
XI. The Nine Situations
XII. The Attack by Fire


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\pooja\AppData\Local\Temp\ipykernel_18312\1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2054.64it/s]


In [7]:
vector_store = FAISS.from_texts(
    chunks,
    embedding=embeddings
)

print("FAISS Vector Store Created")

FAISS Vector Store Created


In [8]:
query = "What is this document about?"

docs = vector_store.similarity_search(
    query,
    k=2
)

for doc in docs:
    print(doc.page_content)
    print("\n-------------------\n")

XI. The Nine Situations
XII. The Attack by Fire
XIII. The Use of SpiesThe Art of War 
32I. Laying Plans167F
168 
Sun Tzǔ said: The art of war is of vital importance to the State. 
It is a matter of life and death, a road either to safety or to ruin. Hence it is a subject of

-------------------

216  
Thus it may be known that the leader of armies is the arbiter of the people’s fate, the man on
whom it depends whether the nation shall be in peace or in peril.216F
217  
40III. Attack by Stratagem

-------------------



In [9]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1267.12it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

In [10]:
context = "\n".join(
    [doc.page_content for doc in docs]
)

print(context)

XI. The Nine Situations
XII. The Attack by Fire
XIII. The Use of SpiesThe Art of War 
32I. Laying Plans167F
168 
Sun Tzǔ said: The art of war is of vital importance to the State. 
It is a matter of life and death, a road either to safety or to ruin. Hence it is a subject of
216  
Thus it may be known that the leader of armies is the arbiter of the people’s fate, the man on
whom it depends whether the nation shall be in peace or in peril.216F
217  
40III. Attack by Stratagem


In [11]:
prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Give a short direct answer.
"""

In [12]:
response = generator(
    prompt,
    max_new_tokens=50
)

answer = response[0]["generated_text"]

print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question using only the context below.

Context:
XI. The Nine Situations
XII. The Attack by Fire
XIII. The Use of SpiesThe Art of War 
32I. Laying Plans167F
168 
Sun Tzǔ said: The art of war is of vital importance to the State. 
It is a matter of life and death, a road either to safety or to ruin. Hence it is a subject of
216  
Thus it may be known that the leader of armies is the arbiter of the people’s fate, the man on
whom it depends whether the nation shall be in peace or in peril.216F
217  
40III. Attack by Stratagem

Question:
What is this document about?

Give a short direct answer.

